In [1]:
from delta import *
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("GoldLayer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
)

from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 3.3.0


In [4]:
BASE = "/home/jovyan/work/delta_lake/silver"

orders       = spark.read.format("delta").load(f"{BASE}/orders")
order_items  = spark.read.format("delta").load(f"{BASE}/order_items")
products     = spark.read.format("delta").load(f"{BASE}/products")
customers    = spark.read.format("delta").load(f"{BASE}/customers")

In [6]:
from pyspark.sql import functions as F

gold_enriched = (
    order_items
    .join(orders,    "order_id")
    .join(products,  "product_id")
    .join(customers, "customer_id", "left")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "product_category_name",
        F.col("product_category_name_english").alias("category_english"),
        "price",
        "freight_value",
        "order_status",
        "order_purchase_timestamp",
        "customer_state",
    )
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("category_english").isNotNull())
)

gold_enriched.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_order_items_enriched")

print("gold_order_items_enriched:", gold_enriched.count(), "rows")
gold_enriched.show(5, truncate=False)

gold_order_items_enriched: 98774 rows
+--------------------------------+--------------------------------+--------------------------------+---------------------+----------------+-----+-------------+------------+------------------------+--------------+
|order_id                        |customer_id                     |product_id                      |product_category_name|category_english|price|freight_value|order_status|order_purchase_timestamp|customer_state|
+--------------------------------+--------------------------------+--------------------------------+---------------------+----------------+-----+-------------+------------+------------------------+--------------+
|20e0101b20700188cadb288126949685|48558a50a7ba1aab61891936d2ca7681|64d0feb1bcf9c7fe7b5dad3271c10910|moveis_decoracao     |furniture_decor |89.18|16.38        |delivered   |2018-01-22 19:22:22     |MG            |
|3a830ecb5338f0ff69822d388b64822e|f8c3d249c98f91b25409df45d4a095e3|bb1865456593db5a35bcc2b23174a09d|esporte_la

In [7]:
gold_basket = (
    gold_enriched
    .groupBy("order_id")
    .agg(F.collect_set("category_english").alias("items"))  # ← đổi collect_list → collect_set
    .filter(F.size("items") >= 2)
)

gold_basket.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_basket")

print("gold_basket:", gold_basket.count(), "đơn hàng hợp lệ")
gold_basket.show(10, truncate=False)

gold_basket: 723 đơn hàng hợp lệ
+--------------------------------+-------------------------------------+
|order_id                        |items                                |
+--------------------------------+-------------------------------------+
|014405982914c2cde2796ddcf0b8703d|[perfumery, sports_leisure]          |
|0420dbc50fc554e303e4b2d6b39063f6|[baby, cool_stuff]                   |
|04cc9ab9d21b11d7aca691cf7facaaa1|[housewares, health_beauty]          |
|06d5cbe68295558fd6c48998db3ea397|[housewares, food_drink]             |
|071be1c98f3a6f467d8e472d94350db8|[stationery, luggage_accessories]    |
|071deb193a4682621c819936d21ae127|[telephony, cool_stuff]              |
|09268d8b25dd31ae78b464efd453d069|[baby, health_beauty]                |
|0cda834be88c6e549a9895efe5e0b49f|[bed_bath_table, home_appliances_2]  |
|122ebb4ea3cb1281899533fc66f7d179|[computers_accessories, garden_tools]|
|133b607354fbff7447ce545c8a0fa407|[furniture_decor, bed_bath_table]    |
+-----------------

In [7]:
print("=== TOP 10 CATEGORIES ===")
(gold_enriched
 .groupBy("category_english")
 .count()
 .orderBy(F.desc("count"))
 .show(10, truncate=False))

print("=== PHÂN BỐ BASKET SIZE ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .groupBy("basket_size")
 .count()
 .orderBy("basket_size")
 .show(15))

print("=== THỐNG KÊ ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .select(
     F.count("order_id").alias("total_orders"),
     F.avg("basket_size").alias("avg_basket_size"),
     F.max("basket_size").alias("max_basket_size"),
 )
 .show())

=== TOP 10 CATEGORIES ===
+----------------------+-----+
|category_english      |count|
+----------------------+-----+
|cama_mesa_banho       |10008|
|beleza_saude          |8830 |
|esporte_lazer         |7663 |
|informatica_acessorios|6723 |
|moveis_decoracao      |6633 |
|utilidades_domesticas |5878 |
|relogios_presentes    |5668 |
|telefonia             |4177 |
|automotivo            |3901 |
|brinquedos            |3895 |
+----------------------+-----+
only showing top 10 rows

=== PHÂN BỐ BASKET SIZE ===
+-----------+-----+
|basket_size|count|
+-----------+-----+
|          2|  708|
|          3|   15|
+-----------+-----+

=== THỐNG KÊ ===
+------------+-----------------+---------------+
|total_orders|  avg_basket_size|max_basket_size|
+------------+-----------------+---------------+
|         723|2.020746887966805|              3|
+------------+-----------------+---------------+



In [8]:
# Đọc lại để verify
check = spark.read.format("delta").load(
    "/home/jovyan/work/delta_lake/gold/gold_basket"
)
print("Verify gold_basket OK — rows:", check.count())

# Xem Delta log
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "/home/jovyan/work/delta_lake/gold/gold_basket")
dt.history(3).select("version", "timestamp", "operation").show()

Verify gold_basket OK — rows: 723
+-------+--------------------+---------+
|version|           timestamp|operation|
+-------+--------------------+---------+
|      0|2026-05-08 10:53:...|    WRITE|
+-------+--------------------+---------+

